# FaceProof 06 — calibration (RQ3, headline finding)

**Day 6.** Measure **ECE** + reliability diagrams and apply **temperature scaling** (`src.calibration`)
fit on `val`. Temperature can't change AUROC/EER (monotonic) — only the honesty of confidence.
Each eval group pairs held-out reals with the fakes of interest (so AUROC is well-defined).

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/Ezed9/faceProof.git"
BRANCH   = "main"
!git clone -b $BRANCH $REPO_URL
%cd faceProof
!pip install -q scikit-learn joblib scipy pyyaml
import sys, os
sys.path.append(os.getcwd())

In [ ]:
from google.colab import drive
from pathlib import Path
import numpy as np, pandas as pd
drive.mount('/content/drive')
ROOT          = Path('/content/drive/MyDrive/faceproof')
CROPS_ROOT    = ROOT / 'data' / 'crops'
FEATURES_ROOT = ROOT / 'models' / 'features'
PROBES_ROOT   = ROOT / 'models' / 'probes'
REPORTS_ROOT  = ROOT / 'reports'
MANIFEST      = ROOT / 'data' / 'manifest.csv'
RESULTS_CSV   = REPORTS_ROOT / 'results.csv'
(REPORTS_ROOT / 'figures').mkdir(parents=True, exist_ok=True)
PROBES_ROOT.mkdir(parents=True, exist_ok=True)

In [ ]:
import csv
RESULT_FIELDS = ['condition','train_gen','test_gen','corruption','strength','seed','metric','value']
def log_result(**row):
    full = {k: row.get(k, '') for k in RESULT_FIELDS}
    new = not RESULTS_CSV.exists()
    with open(RESULTS_CSV, 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=RESULT_FIELDS)
        if new: w.writeheader()
        w.writerow(full)

## 2. Load manifest, features, saved probes

In [ ]:
import joblib
df=pd.read_csv(MANIFEST); y=df['label'].values
X={'c4_clip':np.load(FEATURES_ROOT/'clip_all.npy'),'c1_resnet':np.load(FEATURES_ROOT/'resnet_all.npy')}
for k,v in X.items(): assert len(v)==len(df)
probes={c:joblib.load(PROBES_ROOT/f'{c}.joblib') for c in X}
real_test=(df['label']==0)&(df['split']=='test_indist')
M={'val':(df['split']=='val').values,
   'stylegan_indist':(df['split']=='test_indist').values,
   'stable_diffusion':(real_test|(df['generator']=='stable_diffusion')).values}

## 3. ECE/EER before vs after temperature scaling

`decision_function` returns the logit; for binary LogisticRegression `sigmoid(logit)==predict_proba[:,1]`, so T=1 reproduces the probe's own probabilities. T is fit on `val` only and reused for all test sets.

In [ ]:
from src.metrics import expected_calibration_error as ece, auroc, eer
from src.calibration import fit_temperature, apply_temperature
saved={}; rows=[]
for cond,feats in X.items():
    clf=probes[cond]; logit=lambda mask: clf.decision_function(feats[mask])
    T=fit_temperature(logit(M['val']), y[M['val']])
    for tg in ['stylegan_indist','stable_diffusion']:
        lo=logit(M[tg]); yy=y[M[tg]]
        p_un=apply_temperature(lo,1.0); p_ca=apply_temperature(lo,T)
        e_un,e_ca=ece(yy,p_un),ece(yy,p_ca); au=auroc(yy,p_un); er=eer(yy,p_un)
        saved[(cond,tg)]=(yy,p_un,p_ca)
        for metric,val in [('AUROC',au),('EER',er),('ECE',e_un),('ECE_tempscaled',e_ca),('temperature',T)]:
            log_result(condition=cond,train_gen='stylegan',test_gen=tg,corruption='none',seed=13,metric=metric,value=round(val,4))
        rows.append({'condition':cond,'test_gen':tg,'AUROC':round(au,4),'EER':round(er,4),
                     'ECE':round(e_un,4),'ECE_temp':round(e_ca,4),'T':round(T,3)})
print(pd.DataFrame(rows).to_string(index=False))

## 4. Reliability diagrams + save arrays for the figures notebook

Y-axis = empirical **fraction positive** per predicted-probability bin (consistent with ECE).

In [ ]:
import matplotlib.pyplot as plt
def reliability(ax,yy,p,title,n_bins=15):
    bins=np.linspace(0,1,n_bins+1); xs=[];ys=[]
    for lo,hi in zip(bins[:-1],bins[1:]):
        m=(p>lo)&(p<=hi)
        if m.sum()==0: continue
        xs.append(p[m].mean()); ys.append(yy[m].mean())   # fraction positive
    ax.plot([0,1],[0,1],'--',c='gray'); ax.plot(xs,ys,marker='o')
    ax.set_title(title,fontsize=9); ax.set_xlabel('predicted P(synthetic)'); ax.set_ylabel('fraction synthetic')
conds=list(X); fig,axes=plt.subplots(len(conds),2,figsize=(8,4*len(conds)))
for i,cond in enumerate(conds):
    for j,tg in enumerate(['stylegan_indist','stable_diffusion']):
        yy,p_un,_=saved[(cond,tg)]; reliability(axes[i,j],yy,p_un,f'{cond}/{tg}')
fig.tight_layout(); fig.savefig(REPORTS_ROOT/'figures'/'reliability.png',dpi=150,bbox_inches='tight'); plt.show()
np.savez(REPORTS_ROOT/'calib_arrays.npz',
         **{f'{c}__{t}__{k}':v for (c,t),(yy,pu,pc) in saved.items()
            for k,v in [('y',yy),('p_un',pu),('p_ca',pc)]})
print('✅ Day 6 gate: AUROC|EER|ECE table + reliability diagrams saved')

---
**Finding:** compare ECE in-dist vs unseen. A jump = H3 (confidently wrong under shift).